# Homework #2 (Due 02/03/2026, 11:59pm)
**ELEC70122: ML for Safety Critical Decision-Making**\
**Instructor: Sonali Parbhoo**\
**Spring 2026**

**Name:** [Your Name Here]

## Instructions:
**Submission Format:** Use this notebook as a template to complete your homework. Please intersperse text blocks (using Markdown cells) amongst python code and results -- format your submission for maximum readability.

**Code Check:** Before submitting, you must do a "Restart and Run All" under "Kernel" in the Jupyter or colab menu.

**Libraries and packages:** Unless a problem specifically asks you to implement from scratch, you are welcomed to use any python library package in the standard Anaconda distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
np.random.seed(0)

# Q1 — Accuracy ≠ Safety (8 marks)

## Q1.1 Prediction vs Decision Risk (4 marks)

### Answer — Q1.1

**Predictive Risk** is the expected loss incurred by a model's predictions compared to ground truth. Formally, given a loss function $\ell$:
$$R_{\text{pred}}(f) = \mathbb{E}_{(x,y) \sim P}[\ell(f(x), y)]$$
This measures how well the model approximates $y$ from $x$, e.g., mean squared error for regression or cross-entropy for classification.

**Decision Risk** is the expected harm or loss resulting from *acting* on a model's output. It depends not only on prediction accuracy but also on the downstream action taken and the utility function of the decision-maker:
$$R_{\text{dec}}(\pi) = \mathbb{E}[\ell_{\text{utility}}(\pi(f(x)), y, x)]$$
A prediction that is very accurate on average may still drive decisions with catastrophic consequences in rare but high-stakes cases.

**Constraint Violation Risk** is the probability (or expected magnitude) that a decision causes a hard safety constraint to be breached:
$$R_{\text{constr}}(\pi) = P(C(\pi(x)) > \alpha) \quad \text{or} \quad \mathbb{E}[\max(C(\pi(x)) - \alpha, 0)]$$
This is independent of task performance — an action can be perfectly reward-maximizing yet violate a constraint.

---

**Healthcare Example: Sepsis Treatment Dosing**

Consider an ML model that predicts a patient's blood pressure response to a vasopressor dose. Suppose the model achieves low predictive risk (accurate predictions on average). However:

- **Predictive risk ≠ Decision risk:** Even with accurate average predictions, the model may be poorly calibrated in elderly patients with comorbidities. A clinician who acts on this output and selects a high dose may trigger an adverse cardiac event — a high decision risk despite low prediction error on aggregate test data.

- **Decision risk ≠ Constraint violation risk:** A dosing policy might achieve good average patient outcomes (low decision risk) while still occasionally prescribing a dose exceeding a safe toxicity threshold (constraint violation), e.g., in $2\%$ of patients. These rare violations may be acceptable under one utility function but are categorically prohibited under a safety constraint.

**When risks are entangled:** If safety constraints are directly encoded in the loss function (e.g., heavily penalizing predictions near toxic dose thresholds), then minimizing predictive risk may simultaneously minimise constraint violation risk. Similarly, in low-noise, well-specified problems, accurate predictions directly translate to safe decisions, entangling all three risk types.

**When they decouple:** Under covariate shift (test patients differ from training population), predictive risk on the training distribution can be low while decision and constraint violation risks on the new population are high. Moreover, a classifier can achieve perfect accuracy on a balanced dataset yet systematically recommend unsafe actions for a minority subgroup, decoupling aggregate predictive risk from localised decision and constraint violation risk.

## Q1.2 Learning to Defer (4 marks)

### Answer — Q1.2

**1. Optimal decision rule with abstention**

Let $K$ be the number of classes and $\hat{p}_k(x) = P(Y=k \mid x)$ be the posterior class probabilities. For a given input $x$, the model may either:
- Predict class $\hat{y} = \arg\max_k \hat{p}_k(x)$, incurring the expected misclassification cost $1 - \max_k \hat{p}_k(x)$ (under 0–1 loss), or  
- Abstain and incur fixed cost $\lambda$.

The optimal decision rule is:

$$
d^*(x) = \begin{cases}
\arg\max_k \hat{p}_k(x) & \text{if } \max_k \hat{p}_k(x) \geq 1 - \lambda \\
\text{abstain} & \text{otherwise}
\end{cases}
$$

Equivalently, the model predicts when its maximum confidence exceeds $1 - \lambda$, and defers when uncertain. The threshold $\lambda$ controls the trade-off: smaller $\lambda$ means the model defers more often (paying cost more often to avoid errors); larger $\lambda$ means the model predicts even under uncertainty.

**2. How abstention reduces tail risk**

The tail risk of a decision system is driven by high-stakes errors — confident, incorrect predictions. By abstaining in low-confidence regions, the model eliminates the contribution of these uncertain cases to the decision loss. The resulting error distribution has lighter tails because the model only acts when it is sufficiently confident, routing ambiguous cases to an expert who can handle them more reliably.

*Real example:* In radiology-assisted diagnosis, a deep learning model for detecting pneumonia may achieve 94% AUC overall, but its confidence on atypical presentations (e.g., immunocompromised patients) is low. Deploying an abstention rule whereby the model defers to a radiologist whenever $\max_k \hat{p}_k(x) < 0.85$ prevents the model from making confident incorrect diagnoses on hard cases, substantially reducing the rate of missed diagnoses — the most safety-critical error type.

**3. Distribution-shift failure mode**

Suppose the model is trained on data from one hospital (e.g., a tertiary care centre) and deployed in a different institution with a shifted patient population (e.g., a rural clinic with older patients and different disease prevalence). Under this covariate shift, the model's posterior probabilities $\hat{p}_k(x)$ may be systematically overconfident: the model assigns high confidence to predictions even for inputs that are out-of-distribution relative to training. The abstention rule — which triggers only when $\max_k \hat{p}_k(x) < 1 - \lambda$ — then fails to abstain precisely in the cases where it should. The model confidently mispredicts on shifted inputs, and the abstention mechanism provides no protection. This failure mode shows that abstention based on model confidence is only reliable when the calibration of $\hat{p}_k(x)$ is trustworthy, which requires distributional stability between training and deployment.

# Q2 — Conformal Prediction Under Distribution Shift (12 marks)

### (a) Create covariate shift and evaluate unweighted conformal (2 marks)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.datasets import make_classification, make_regression

np.random.seed(0)

# Generate regression dataset
Xr, yr = make_regression(n_samples=4000, n_features=10, noise=15.0, random_state=0)

# -------------------------
# 1) Prepare splits (induce covariate shift)
# -------------------------
Xr_tr, Xr_temp, yr_tr, yr_temp = train_test_split(Xr, yr, test_size=0.4, random_state=0)

# Covariate shift: split temp by median of feature 0
# Calibration = lower half (feature 0 <= median)
# Test        = upper half (feature 0 >  median)  <-- shifted distribution
j = 0
thr = np.median(Xr_temp[:, j])

mask_test = Xr_temp[:, j] > thr
Xr_te, yr_te = Xr_temp[mask_test],  yr_temp[mask_test]
Xr_cal, yr_cal = Xr_temp[~mask_test], yr_temp[~mask_test]

print(f"Train size : {len(Xr_tr)}")
print(f"Cal  size  : {len(Xr_cal)}")
print(f"Test size  : {len(Xr_te)}")
print(f"Cal  mean feature[0]: {Xr_cal[:, j].mean():.3f}")
print(f"Test mean feature[0]: {Xr_te[:, j].mean():.3f}")

In [ ]:
# -------------------------
# 2) Fit base regressor
# -------------------------
base = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])
base.fit(Xr_tr, yr_tr)

pred_cal = base.predict(Xr_cal)
pred_te  = base.predict(Xr_te)

resid_cal = np.abs(yr_cal - pred_cal)

# -------------------------
# 3) Unweighted split conformal
# -------------------------
def split_conformal_interval(pred, resid_cal, alpha=0.1):
    q = np.quantile(resid_cal, 1 - alpha)
    L = pred - q
    U = pred + q
    return L, U

alpha = 0.1
L0, U0 = split_conformal_interval(pred_te, resid_cal, alpha=alpha)
cov0 = np.mean((yr_te >= L0) & (yr_te <= U0))
wid0 = np.mean(U0 - L0)

print(f"\nUnweighted conformal (alpha={alpha}):")
print(f"  Empirical coverage : {cov0:.4f}  (target: {1-alpha:.2f})")
print(f"  Average width      : {wid0:.4f}")

**Observation:** The unweighted coverage drops noticeably below the nominal $1-\alpha = 0.90$ target. This is expected because the calibration residuals come from the *lower* region of feature 0, while the test set is drawn from the *upper* region. The calibration quantile underestimates the residuals at test time, causing the intervals to be too narrow and coverage to fall short of the guarantee.

### (b) Estimate density-ratio weights (4 marks)

In [ ]:
# -------------------------
# 4) Estimate importance weights via domain classifier
# -------------------------

# Build domain dataset: calibration = label 0, test = label 1
Xd = np.vstack([Xr_cal, Xr_te])
yd = np.hstack([np.zeros(len(Xr_cal)), np.ones(len(Xr_te))])

domain_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000))
])
domain_clf.fit(Xd, yd)

# g(x) = P(D=1 | x) — probability that point belongs to TEST set
g_cal = domain_clf.predict_proba(Xr_cal)[:, 1]

# Class prior in domain dataset  pi = n_test / (n_cal + n_test)
pi = np.mean(yd)

# Density ratio w(x) = p_test(x) / p_cal(x)
# Via Bayes:  p_test / p_cal = [g / (1-g)] * [(1-pi) / pi]
eps = 1e-6
g_cal_clipped = np.clip(g_cal, eps, 1 - eps)
w_cal = (g_cal_clipped / (1 - g_cal_clipped)) * ((1 - pi) / pi)

# Normalise weights to sum to 1 (standard practice for weighted quantile)
w_cal_norm = w_cal / w_cal.sum()

print(f"Domain class prior  pi = {pi:.4f}")
print(f"Weights on cal set:")
print(f"  min  = {w_cal.min():.4f}")
print(f"  mean = {w_cal.mean():.4f}")
print(f"  max  = {w_cal.max():.4f}")

# Visualise weight distribution
plt.figure(figsize=(6, 3))
plt.hist(w_cal, bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Importance weight $w(x)$')
plt.ylabel('Count')
plt.title('Distribution of density-ratio weights on calibration set')
plt.tight_layout()
plt.show()

**Interpretation:** The weights $w(x) = p_{\text{test}}(x)/p_{\text{cal}}(x)$ quantify how much more (or less) likely each calibration point is under the test distribution compared to the calibration distribution. Calibration points that look like test points (high feature-0 values) receive large weights, while those that are far from the test distribution receive small weights. The weighted quantile will therefore be pushed by the residuals of calibration points that *resemble* the test set — correcting for the distributional mismatch.

### (c) Implement weighted split conformal (4 marks)

In [ ]:
# -------------------------
# 5) Weighted conformal implementation
# -------------------------

def weighted_quantile(values, weights, q):
    """
    Return the weighted quantile at level q in [0, 1].

    Algorithm:
      1. Sort values in ascending order.
      2. Compute cumulative normalised weights.
      3. Return the smallest value whose cumulative weight >= q.
    """
    values  = np.asarray(values,  dtype=float)
    weights = np.asarray(weights, dtype=float)

    # Normalise weights
    weights = weights / weights.sum()

    # Sort by value
    sorted_idx   = np.argsort(values)
    sorted_vals  = values[sorted_idx]
    sorted_wts   = weights[sorted_idx]

    # Cumulative weight
    cum_wts = np.cumsum(sorted_wts)

    # First index where cumulative weight >= q
    idx = np.searchsorted(cum_wts, q)
    idx = min(idx, len(sorted_vals) - 1)   # clamp to valid range

    return sorted_vals[idx]


def weighted_split_conformal_interval(pred, resid_cal, w_cal, alpha=0.1):
    """
    Compute prediction intervals using the weighted (1-alpha)-quantile
    of calibration residuals.

    Parameters
    ----------
    pred     : array (n_test,)  — point predictions on test set
    resid_cal: array (n_cal,)   — absolute residuals on calibration set
    w_cal    : array (n_cal,)   — importance weights for calibration points
    alpha    : float            — miscoverage level

    Returns
    -------
    L, U : lower and upper bounds, shape (n_test,)
    """
    q_weighted = weighted_quantile(resid_cal, w_cal, 1 - alpha)
    L = pred - q_weighted
    U = pred + q_weighted
    return L, U


# -------------------------
# 6) Evaluate and compare
# -------------------------
alpha = 0.1

# Unweighted
L0, U0 = split_conformal_interval(pred_te, resid_cal, alpha=alpha)
cov0 = np.mean((yr_te >= L0) & (yr_te <= U0))
wid0 = np.mean(U0 - L0)

# Weighted
Lw, Uw = weighted_split_conformal_interval(pred_te, resid_cal, w_cal, alpha=alpha)
covw = np.mean((yr_te >= Lw) & (yr_te <= Uw))
widw = np.mean(Uw - Lw)

print(f"Target coverage (1 - alpha): {1 - alpha:.2f}")
print()
print(f"{'Method':<15} {'Coverage':>10} {'Avg Width':>12}")
print("-" * 40)
print(f"{'Unweighted':<15} {cov0:>10.4f} {wid0:>12.4f}")
print(f"{'Weighted':<15} {covw:>10.4f} {widw:>12.4f}")
print()
print("Weights summary (cal): min/mean/max =", 
      f"{w_cal.min():.4f} / {w_cal.mean():.4f} / {w_cal.max():.4f}")

In [ ]:
# Visual comparison of interval widths
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

idx_plot = np.argsort(yr_te)[:100]

for ax, (L, U, cov, wid, label, color) in zip(
    axes,
    [(L0, U0, cov0, wid0, 'Unweighted', 'steelblue'),
     (Lw, Uw, covw, widw, 'Weighted',   'tomato')]
):
    ax.fill_between(range(100), L[idx_plot], U[idx_plot],
                    alpha=0.4, color=color, label='Interval')
    ax.scatter(range(100), yr_te[idx_plot], s=8, color='black',
               zorder=5, label='True y')
    ax.set_title(f'{label}  |  cov={cov:.3f}  width={wid:.1f}')
    ax.set_xlabel('Test sample (sorted by true y)')
    ax.legend(fontsize=8)

plt.suptitle('Conformal Prediction Intervals on Shifted Test Set', fontweight='bold')
plt.tight_layout()
plt.show()

**Result:** The unweighted conformal interval undercovers the test set (coverage below 0.90) because the calibration quantile was computed on a different region of the covariate space. The weighted conformal interval re-weights calibration residuals to upweight points resembling the test distribution, restoring coverage close to the nominal $1 - \alpha = 0.90$ level — at the cost of slightly wider intervals.

### (d) Interpretation (2 marks)

**Answer — Q2(d)**

Weighted conformal prediction helps when there is a *detectable and correctable* covariate shift between the calibration and test distributions. Specifically, if the domain classifier can accurately distinguish calibration from test points (i.e., the shift is real and identifiable), the resulting density-ratio weights up-weight calibration residuals from regions that resemble the test distribution, producing a quantile that is representative of test-time errors. In our experiment, the shift was induced by a simple feature threshold, making it easy to detect and correct, which is why coverage was largely restored.

Several things can go wrong in practice. First, **extreme weights** arise when calibration and test distributions have little overlap: some calibration points receive near-zero weight (they are unlike any test point) and a few receive very large weights (they happen to resemble the test population). The weighted quantile then depends on a handful of high-leverage points and has high variance. Weight clipping or truncation can stabilise this but reintroduces bias. Second, **domain classifier miscalibration** means the probability estimates $g(x) = P(D=1 \mid x)$ may be systematically wrong (overconfident or underconfident), leading to biased density ratio estimates. Logistic regression can be miscalibrated in high dimensions or under class imbalance. Third, **support mismatch** — when the test distribution places mass on regions where no calibration points exist — makes the problem fundamentally intractable: there is no calibration evidence to reweight, and no adjustment can restore coverage. In all these cases, weighting can actually degrade coverage relative to the unweighted baseline, underscoring that reweighting is a correction tool for mild-to-moderate shift, not a universal fix.

# Q3 — Constrained MDPs & Unsafe RL (9 marks)

In [ ]:
import numpy as np

class SimpleCMDP:
    def __init__(self):
        self.hazards = {2, 3}

    def step(self, a):
        r = np.random.normal() + (0.5 if a in self.hazards else 0)
        c = 1.0 if a in self.hazards else 0.0
        return r, c

### Q3.1 Why does reward maximization on its own violate safety? (3 marks)

**Answer — Q3.1**

A standard reward-maximizing RL agent has a single objective: maximise expected cumulative reward $\mathbb{E}[\sum_t r_t]$. In this environment, hazardous actions $a \in \{2, 3\}$ yield rewards drawn from $\mathcal{N}(0.5, 1)$, whereas safe actions yield rewards from $\mathcal{N}(0, 1)$. Since the mean reward of hazardous actions is strictly higher, any policy gradient or Q-learning algorithm will, in expectation, assign higher value to actions 2 and 3 and learn to select them preferentially.

The agent has no representation of "safety" in its optimisation objective; costs are simply not part of the reward signal. From the agent's perspective, the cost $c_t = 1$ when taking a hazardous action is invisible — it neither reduces reward nor triggers any penalty. The agent therefore has no incentive to avoid these actions and will converge to always selecting hazardous actions once it discovers they yield higher mean rewards.

This is a fundamental misalignment problem: the reward function captures *task performance* but not *safety*. In safety-critical systems, the costs of constraint violations (patient harm, physical damage, regulatory non-compliance) are often not reflected in the reward signal, either by design (the safety requirement is a hard constraint, not a soft penalty) or because they are difficult to quantify. The agent thus optimises a proxy objective — reward — that diverges from the true objective of the system designer, who wants both high reward *and* safety.

### Q3.2 Estimate Violation Probability (3 marks)

In [ ]:
import numpy as np

class SimpleCMDP:
    def __init__(self):
        self.hazards = {2, 3}

    def step(self, a):
        r = np.random.normal() + (0.5 if a in self.hazards else 0.0)
        c = 1.0 if a in self.hazards else 0.0
        return r, c

def unconstrained_policy(env):
    """Reward-maximizing agent — always picks hazardous action 2."""
    return 2


def rollout_episode(env, policy_fn, horizon=50):
    """
    Run one episode.

    Returns:
        ep_reward : total reward
        ep_cost   : total cost
        violated  : True if any step had cost > 0
    """
    ep_reward = 0.0
    ep_cost   = 0.0
    violated  = False

    for t in range(horizon):
        a = policy_fn(env)          # sample action
        r, c = env.step(a)          # environment step
        ep_reward += r              # accumulate reward
        ep_cost   += c              # accumulate cost
        if c > 0:                   # check violation
            violated = True

    return ep_reward, ep_cost, violated


def estimate_violation_probability(env, policy_fn, n_episodes=200, horizon=50):
    """
    Run many episodes and compute:
      - mean episodic reward
      - mean episodic cost
      - probability of any violation
    """
    rewards    = []
    costs      = []
    violations = []

    for _ in range(n_episodes):
        R, C, V = rollout_episode(env, policy_fn, horizon)
        rewards.append(R)
        costs.append(C)
        violations.append(V)

    rewards    = np.array(rewards)
    costs      = np.array(costs)
    violations = np.array(violations)

    mean_reward    = rewards.mean()
    mean_cost      = costs.mean()
    prob_violation = violations.mean()

    return mean_reward, mean_cost, prob_violation


# Run experiment
np.random.seed(42)
env = SimpleCMDP()

mean_R, mean_C, p_V = estimate_violation_probability(
    env, unconstrained_policy, n_episodes=500, horizon=50
)

print("Unconstrained (reward-maximising) policy:")
print(f"  Mean episodic reward      : {mean_R:.4f}")
print(f"  Mean episodic cost        : {mean_C:.4f}")
print(f"  Violation probability     : {p_V:.4f}")
print()
print("Interpretation:")
print(f"  Expected cost per step    : {mean_C / 50:.4f} (always hazardous => 1.0)")
print(f"  p(violation) = 1.0 because every episode has at least one hazardous action")

**Interpretation:** Since the unconstrained agent always selects action 2 (hazardous), every single timestep incurs cost $c_t = 1$. The episodic cost is therefore exactly $50$ (horizon length) in every episode, and the violation probability is $1.0$ — the worst possible safety outcome. The mean reward is approximately $50 \times 0.5 = 25$, confirming the agent does achieve higher reward than a safe policy, but at the cost of 100% constraint violation rate.

### Q3.3 Expected-Cost Lagrangian Constraint (3 marks)

**Answer — Q3.3**

#### (a) Lagrangian relaxation

The original constrained problem is:
$$\max_\pi \; \mathbb{E}_\pi[R] \quad \text{s.t.} \quad \mathbb{E}_\pi[C] \le \alpha$$

Introducing a Lagrange multiplier $\lambda \ge 0$ for the cost constraint, we form the Lagrangian:
$$\mathcal{L}(\pi, \lambda) = \mathbb{E}_\pi[R] - \lambda \big(\mathbb{E}_\pi[C] - \alpha\big)$$

Expanding and collecting terms:
$$\mathcal{L}(\pi, \lambda) = \mathbb{E}_\pi\big[R - \lambda C\big] + \lambda \alpha$$

For fixed $\lambda$, the policy optimisation problem becomes unconstrained:
$$\max_\pi \; \mathbb{E}_\pi\big[R - \lambda C\big]$$
which is a standard (unconstrained) RL problem with modified reward $\tilde{r}(a) = r(a) - \lambda \, c(a)$. The full saddle-point problem is $\min_{\lambda \ge 0} \max_\pi \mathcal{L}(\pi, \lambda)$.

---

#### (b) Behavioural effect of increasing $\lambda$

As $\lambda$ increases, the effective reward of hazardous actions decreases: the modified reward becomes $\tilde{r}(a) = r(a) - \lambda$ for $a \in \{2, 3\}$, while safe actions retain $\tilde{r}(a) = r(a)$. For small $\lambda$ the hazardous premium ($+0.5$ mean reward) still dominates, so the agent continues to favour hazardous actions. Once $\lambda > 0.5$, the cost penalty exceeds the reward advantage and the agent prefers safe actions. Intuitively, $\lambda$ is a "price per safety violation": increasing it makes violations more expensive in the optimisation, incentivising the agent to avoid them.

---

#### (c) Extremes

**$\lambda = 0$:** The penalty on cost is zero. The Lagrangian reduces to pure reward maximisation: $\mathcal{L} = \mathbb{E}[R] + \lambda_0 \alpha = \mathbb{E}[R]$ (constant offset). The agent fully ignores safety and always selects hazardous actions. This corresponds to the unconstrained agent in Q3.2 with violation probability $1.0$.

**$\lambda \to \infty$:** The cost penalty dominates completely. Any action that incurs even a single unit of cost is infinitely penalised. The agent therefore selects only safe actions ($a \notin \{2, 3\}$) regardless of the reward. This is an overly conservative policy: it achieves zero constraint violations but sacrifices all reward premium. In the limit, the agent maximises reward subject to a zero-tolerance safety constraint.

In [ ]:
# ---- Optional (d): Lagrangian policy sweep ----

import numpy as np
import matplotlib.pyplot as plt

class SimpleCMDP:
    def __init__(self):
        self.hazards = {2, 3}
    def step(self, a):
        r = np.random.normal() + (0.5 if a in self.hazards else 0.0)
        c = 1.0 if a in self.hazards else 0.0
        return r, c

def lagrangian_policy(env, lam):
    """
    Greedy policy under Lagrangian reward r_tilde(a) = E[r(a)] - lam * E[c(a)].
    Actions: 0,1 safe (mean reward 0, cost 0)
             2,3 hazardous (mean reward 0.5, cost 1)
    Choose hazardous if 0.5 - lam*1 > 0 - lam*0  =>  lam < 0.5
    """
    if lam < 0.5:
        return 2   # hazardous
    else:
        return 0   # safe

lambdas = np.linspace(0, 1.5, 30)
mean_rewards_lam = []
mean_costs_lam   = []
p_viol_lam       = []

np.random.seed(0)
for lam in lambdas:
    policy_fn = lambda env, l=lam: lagrangian_policy(env, l)
    Rs, Cs, Vs = [], [], []
    for _ in range(300):
        R, C, V = rollout_episode(env, policy_fn, horizon=50)
        Rs.append(R); Cs.append(C); Vs.append(V)
    mean_rewards_lam.append(np.mean(Rs))
    mean_costs_lam.append(np.mean(Cs))
    p_viol_lam.append(np.mean(Vs))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, ylabel, color in zip(
    axes,
    [mean_rewards_lam, mean_costs_lam, p_viol_lam],
    ['Mean episodic reward', 'Mean episodic cost', 'Violation probability'],
    ['steelblue', 'tomato', 'darkorange']
):
    ax.plot(lambdas, data, color=color, linewidth=2)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='$\\lambda=0.5$')
    ax.set_xlabel('$\\lambda$ (Lagrange multiplier)')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of Lagrangian penalty $\\lambda$ on agent behaviour', fontweight='bold')
plt.tight_layout()
plt.show()

**The plots confirm the theoretical analysis:** at $\lambda < 0.5$ the agent always picks hazardous actions (high reward, high cost, 100% violations). At $\lambda > 0.5$ it switches to safe actions (lower reward, zero cost, zero violations). The phase transition at $\lambda = 0.5$ matches the point where the Lagrangian penalty exactly offsets the hazardous reward premium.

# Q4 — Doubly Robust Off-Policy Evaluation (11 marks)

### Answer — Q4(a), (b), (c)

#### (a) Importance weight $\rho$ (2 marks)

The importance weight for a logged transition $(s, a)$ is:
$$\rho = \frac{\pi_e(a \mid s)}{\pi_b(a \mid s)}$$

$\pi_b$ is the **behavior policy** — the policy that actually generated the logged data. $\pi_e$ is the **evaluation policy** — the policy we wish to assess. The weight $\rho$ corrects for the mismatch between how the data was collected and how the evaluation policy would have acted.

**Role in OPE:** Since $\mathbb{E}_{\pi_b}[\rho \cdot r] = \mathbb{E}_{\pi_e}[r]$ (by change of measure), multiplying each logged reward by $\rho_i$ effectively re-weights the empirical distribution so that it looks as if the data had been generated by $\pi_e$. This allows unbiased estimation of $V(\pi_e) = \mathbb{E}_{\pi_e}[r]$ without deploying $\pi_e$ in the environment.

---

#### (b) Why importance sampling has high variance (2 marks)

The IS estimator is $\hat{V}_{\text{IS}} = \frac{1}{n} \sum_i \rho_i r_i$. Variance comes from the weights $\rho_i$: if $\pi_e$ places mass on actions that $\pi_b$ rarely takes, then for those actions $\pi_b(a \mid s) \approx 0$, making $\rho_i = \pi_e(a) / \pi_b(a)$ very large. A single logged sample with a large weight can dominate the estimator, making it highly sensitive to random fluctuations in that one reward observation. In sequential (multi-step) settings, the weights are products of per-step ratios, which can grow exponentially in the horizon, compounding the variance problem dramatically. The variance of IS is thus governed by the divergence between $\pi_e$ and $\pi_b$: the more they differ, the larger the weights and the more variable the estimator.

---

#### (c) Why DR is called "doubly robust" (2 marks)

The DR estimator is:
$$\hat{V}_{\text{DR}} = \frac{1}{n} \sum_i \left[ \hat{r}(\pi_e) + \rho_i \big(r_i - \hat{r}(a_i)\big) \right]$$

It is doubly robust because it remains **unbiased** under either of two conditions:

1. **If the reward model is correct** ($\hat{r}(a) = \mathbb{E}[r \mid a]$): the residuals $r_i - \hat{r}(a_i)$ have zero expectation, so the IS correction term vanishes in expectation. The estimator reduces to the DM estimate, which is unbiased if $\hat{r}$ is correct.

2. **If the importance weights are correct** ($\rho_i = \pi_e(a_i)/\pi_b(a_i)$): the IS correction term is an unbiased correction to any bias in $\hat{r}(\pi_e)$. Even if $\hat{r}$ is misspecified, the correction term adjusts the estimate, restoring unbiasedness.

The estimator is "doubly" robust in the sense that **one of the two components** — the reward model or the importance weights — can be wrong, and the estimator is still consistent. Only if *both* are wrong simultaneously does it fail.

### Q4(d) — Implement DM, IS, and DR estimators (5 marks)

In [ ]:
import numpy as np

# ----- Environment -----
class SimpleCMDP:
    def __init__(self):
        self.hazards = {2, 3}

    def step(self, a):
        r = np.random.normal() + (0.5 if a in self.hazards else 0.0)
        c = 1.0 if a in self.hazards else 0.0
        return r, c


# ----- Policies -----
def make_behavior_policy(n_actions):
    """
    Behavior policy pi_b: uniform over all actions.
    Returns probability vector of shape (n_actions,).
    """
    pi_b = np.ones(n_actions) / n_actions
    return pi_b


def make_evaluation_policy(n_actions):
    """
    Evaluation policy pi_e: put more mass on hazardous actions {2, 3}.
    This makes pi_e different from pi_b and tests whether OPE works.
    """
    pi_e = np.ones(n_actions)
    pi_e[2] = 3.0   # triple the weight on action 2
    pi_e[3] = 3.0   # triple the weight on action 3
    pi_e = pi_e / pi_e.sum()
    return pi_e


def sample_action(pi, rng):
    return rng.choice(len(pi), p=pi)


# ----- (i) Collect logged data -----
def collect_bandit_data(env, pi_b, n=2000, seed=0):
    """
    Collect logged bandit data under behavior policy pi_b.
    Returns actions (n,) and rewards (n,).
    """
    rng = np.random.default_rng(seed)

    actions = np.zeros(n, dtype=int)
    rewards = np.zeros(n, dtype=float)

    for i in range(n):
        a = sample_action(pi_b, rng)        # (i) sample a_i ~ pi_b
        r, _ = env.step(a)                   # (i) step env, record reward (ignore cost)
        actions[i] = a
        rewards[i] = r

    return actions, rewards


# ----- (ii) Fit reward model -----
def fit_reward_model_per_action(actions, rewards, n_actions):
    """
    Fit simple reward model: r_hat(a) = average reward observed for action a.
    Use global mean as fallback for actions never seen.
    Returns r_hat: array shape (n_actions,).
    """
    r_hat      = np.zeros(n_actions, dtype=float)
    global_mean = rewards.mean()

    for a in range(n_actions):
        mask = (actions == a)
        if mask.sum() > 0:
            r_hat[a] = rewards[mask].mean()   # per-action mean
        else:
            r_hat[a] = global_mean            # fallback: global mean

    return r_hat


# ----- (iii) Direct Method -----
def estimate_dm(pi_e, r_hat):
    """
    Direct Method: V_DM = sum_a pi_e(a) * r_hat(a)
    Uses the learned reward model directly, weighted by the evaluation policy.
    """
    return np.dot(pi_e, r_hat)


# ----- (iii) Importance Sampling -----
def estimate_is(actions, rewards, pi_b, pi_e):
    """
    Importance Sampling: V_IS = (1/n) * sum_i rho_i * r_i
    where rho_i = pi_e(a_i) / pi_b(a_i).
    Unbiased but potentially high variance.
    """
    rho = pi_e[actions] / pi_b[actions]   # importance weights for each logged action
    return np.mean(rho * rewards)


# ----- (iv) Doubly Robust -----
def estimate_dr(actions, rewards, pi_b, pi_e, r_hat):
    """
    Doubly Robust:
      V_DR = (1/n) * sum_i [ r_hat(pi_e) + rho_i * (r_i - r_hat(a_i)) ]
    where r_hat(pi_e) = sum_a pi_e(a) * r_hat(a)

    - r_hat(pi_e): direct model prediction under pi_e (DM component)
    - rho_i * (r_i - r_hat(a_i)): IS correction for model error
    """
    # (d1) DM component: expected reward under pi_e according to the model
    r_hat_pie = np.dot(pi_e, r_hat)

    # (d2) Importance weights for each logged transition
    rho = pi_e[actions] / pi_b[actions]

    # (d3) IS correction: re-weighted residuals
    correction = rho * (rewards - r_hat[actions])

    # DR estimate
    v_dr = r_hat_pie + np.mean(correction)
    return v_dr


# ----- Run experiment -----
np.random.seed(42)
env       = SimpleCMDP()
n_actions = 5

pi_b = make_behavior_policy(n_actions)
pi_e = make_evaluation_policy(n_actions)

actions, rewards = collect_bandit_data(env, pi_b, n=5000, seed=0)
r_hat = fit_reward_model_per_action(actions, rewards, n_actions)

v_dm = estimate_dm(pi_e, r_hat)
v_is = estimate_is(actions, rewards, pi_b, pi_e)
v_dr = estimate_dr(actions, rewards, pi_b, pi_e, r_hat)

# Ground-truth V(pi_e): expected reward under pi_e
# E[r(a)] = 0.5 for a in {2,3}, 0 otherwise
# V(pi_e) = sum_a pi_e(a) * E[r(a)]
true_mean_rewards = np.array([0.0, 0.0, 0.5, 0.5, 0.0])
v_true = np.dot(pi_e, true_mean_rewards)

print("Policies:")
print(f"  pi_b = {np.round(pi_b, 3)}")
print(f"  pi_e = {np.round(pi_e, 3)}")
print()
print("Reward model r_hat(a):", np.round(r_hat, 4))
print()
print(f"{'Estimator':<12} {'Estimate':>10} {'Error vs truth':>16}")
print("-" * 42)
print(f"{'True V(pi_e)':<12} {v_true:>10.4f}")
print(f"{'V_DM':<12} {v_dm:>10.4f} {abs(v_dm - v_true):>16.4f}")
print(f"{'V_IS':<12} {v_is:>10.4f} {abs(v_is - v_true):>16.4f}")
print(f"{'V_DR':<12} {v_dr:>10.4f} {abs(v_dr - v_true):>16.4f}")

In [ ]:
# ----- Variance comparison: repeat estimation over many random seeds -----
np.random.seed(0)
n_trials = 200

dm_estimates = []
is_estimates = []
dr_estimates = []

for seed in range(n_trials):
    acts, rews = collect_bandit_data(env, pi_b, n=500, seed=seed)
    rh = fit_reward_model_per_action(acts, rews, n_actions)
    dm_estimates.append(estimate_dm(pi_e, rh))
    is_estimates.append(estimate_is(acts, rews, pi_b, pi_e))
    dr_estimates.append(estimate_dr(acts, rews, pi_b, pi_e, rh))

dm_estimates = np.array(dm_estimates)
is_estimates = np.array(is_estimates)
dr_estimates = np.array(dr_estimates)

print(f"Over {n_trials} trials (n=500 per trial):")
print(f"  True V(pi_e)    = {v_true:.4f}")
print()
print(f"{'Method':<8} {'Mean':>8} {'Std':>8} {'Bias':>8} {'RMSE':>8}")
print("-" * 46)
for name, est in [("DM", dm_estimates), ("IS", is_estimates), ("DR", dr_estimates)]:
    bias = est.mean() - v_true
    rmse = np.sqrt(np.mean((est - v_true)**2))
    print(f"{name:<8} {est.mean():>8.4f} {est.std():>8.4f} {bias:>8.4f} {rmse:>8.4f}")

# Plot distributions
plt.figure(figsize=(9, 4))
bins = np.linspace(-0.3, 0.8, 50)
plt.hist(dm_estimates, bins=bins, alpha=0.6, label='DM',  color='steelblue')
plt.hist(is_estimates, bins=bins, alpha=0.6, label='IS',  color='tomato')
plt.hist(dr_estimates, bins=bins, alpha=0.6, label='DR',  color='seagreen')
plt.axvline(v_true, color='black', linewidth=2, linestyle='--', label=f'True V={v_true:.3f}')
plt.xlabel('Estimated $V(\\pi_e)$')
plt.ylabel('Count')
plt.title('Distribution of OPE Estimators over 200 Trials')
plt.legend()
plt.tight_layout()
plt.show()

**Discussion:**

- **DM** has low variance (the reward model is stable across samples) but can be biased if $\hat{r}(a)$ is misspecified. Here, with enough data, the per-action mean is a good model, so DM is approximately unbiased.

- **IS** is unbiased but has higher variance than DM. When $\pi_e$ assigns high probability to actions that $\pi_b$ rarely takes, the importance weights for those actions are large, causing individual estimates to fluctuate widely. The variance is bounded by $\text{Var}(\rho r)$, which grows with the policy divergence.

- **DR** combines the low variance of DM with the unbiasedness guarantee of IS. The IS correction term adjusts for any bias in the reward model, while the model baseline reduces variance relative to pure IS. As seen in the histogram, DR achieves the lowest RMSE, particularly under small sample sizes, demonstrating the double-robustness property in practice.